In [1]:
# 📦 Imports
import pandas as pd
import random
import sys
import os

# 🔧 Set up path for src
sys.path.append(os.path.abspath('../src'))

# 🧠 Load Recommenders
from cf_model import CFRecommender
from content_model import ContentRecommender
from embedding_model import EmbeddingRecommender
from hybrid_model import HybridRecommender
from reranker import Reranker  # If needed

# 📁 Load Processed Data
df = pd.read_pickle('../data/processed_playlists.pkl')
df_tracks = pd.read_pickle('../data/processed_tracks.pkl')

# ✅ Initialize Recommenders
cf = CFRecommender()
cf.prepare_matrix(df)
cf.train()

content = ContentRecommender()
content.prepare(df_tracks)

embedding = EmbeddingRecommender(vector_size=64, window=5, min_count=1)
playlists = embedding.prepare_training_data(df)
embedding.train(playlists)

# 🔀 Initialize Hybrid Recommender
hybrid = HybridRecommender(cf_model=cf, content_model=content, embedding_model=embedding, weights=(0.4, 0.3, 0.3))

# 🎧 Choose a random playlist
example_pid = df['playlist_id'].sample(1).iloc[0]
playlist_tracks = df[df['playlist_id'] == example_pid]['track_uri'].tolist()

print("🎧 Example Playlist ID:", example_pid)
print("📀 Original Playlist Tracks:")
display(df_tracks[df_tracks['track_uri'].isin(playlist_tracks)][['track_name', 'artist_name']])

# 🔍 Hybrid Recommendation with Scores
recommendations_with_scores = hybrid.recommend_tracks(example_pid, playlist_tracks, top_n=10)

# 📊 Unpack track URIs and scores
recommended_uris = [track for track, _ in recommendations_with_scores]
recommended_scores = [score for _, score in recommendations_with_scores]

print("\n🎯 Hybrid Recommendations (Before Reranking):")
display(df_tracks[df_tracks['track_uri'].isin(recommended_uris)][['track_name', 'artist_name']])

# 🧪 Optional: Apply Reranker
# reranker = Reranker(df_tracks)
# reranked_uris = reranker.rerank(recommended_uris, playlist_tracks)
# print("\n🔁 Reranked Recommendations:")
# display(df_tracks[df_tracks['track_uri'].isin(reranked_uris)][['track_name', 'artist_name']])


/opt/anaconda3/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


🔄 Preparing user-item matrix...
Training collaborative filtering model...


python(73664) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/20 [00:00<?, ?it/s]

🎧 Example Playlist ID: 151780
📀 Original Playlist Tracks:


,track_name,artist_name



🎯 Hybrid Recommendations (Before Reranking):


,track_name,artist_name
550348,Ar Éireann Ní Neosainn Cé hÍ,The High Kings
2326957,Irish Rover,The High Kings
3077203,Galway To Graceland,The High Kings
5757590,Great When You're High,Kottonmouth Kings


In [2]:
# --- 📦 Imports for reranker ---
from reranker import Reranker

# --- 🔧 Build popularity and artist map (only once) ---
track_popularity = df['track_uri'].value_counts()
track_artist_map = pd.Series(df_tracks.artist_name.values, index=df_tracks.track_uri).to_dict()

# --- 🔁 Initialize reranker ---
reranker = Reranker(
    track_popularity=track_popularity,
    popularity_weight=0.7,
    diversity_weight=0.3
)
reranker.set_artist_map(track_artist_map)

# --- 🎯 Get hybrid recommendations with scores ---
recommendations_with_scores = hybrid.recommend_tracks(example_pid, playlist_tracks, top_n=15)

# --- 🔁 Rerank using real scores ---
recommended_uris = [track for track, score in recommendations_with_scores]
scores = [score for track, score in recommendations_with_scores]

reranked_uris = reranker.rerank(recommended_uris, scores)

# --- 📃 Show Reranked Tracks ---
print("\n🔥 Reranked Recommendations:")
for uri in reranked_uris[:10]:
    name = df_tracks.loc[df_tracks['track_uri'] == uri, 'track_name'].values[0]
    artist = df_tracks.loc[df_tracks['track_uri'] == uri, 'artist_name'].values[0]
    print(f"✅ {name} by {artist}")



🔥 Reranked Recommendations:


IndexError: index 0 is out of bounds for axis 0 with size 0

In [5]:
import pickle
import os

# Create models directory if not exists
os.makedirs('../models', exist_ok=True)

# --- Save CF model ---
cf.model.save('../models/cf_model.model')
print("✅ CF model saved.")

# --- Save Content model ---
with open('../models/content_model.pkl', 'wb') as f:
    pickle.dump({
        'tfidf_matrix': content.tfidf_matrix,
        'track_index': content.track_idx_map
    }, f)
print("✅ Content model saved.")

# --- Save Embedding model ---
embedding.model.save('../models/embedding_model.model')
print("✅ Embedding model saved.")

# --- Save Hybrid config (weights) ---
with open('../models/hybrid_config.pkl', 'wb') as f:
    pickle.dump({'weights': hybrid.weights}, f)
print("✅ Hybrid config saved.")


✅ CF model saved.
✅ Content model saved.
✅ Embedding model saved.
✅ Hybrid config saved.


In [4]:
import pickle
from implicit.als import AlternatingLeastSquares
from gensim.models import Word2Vec

# Load CF model
cf_model = AlternatingLeastSquares()
cf_model.load('../models/cf_model.model')

# Load content model
with open('../models/content_model.pkl', 'rb') as f:
    data = pickle.load(f)
content = ContentRecommender()
content.tfidf_matrix = data['tfidf_matrix']
content.track_idx_map = data['track_index']

# Load embedding model
embedding_model = Word2Vec.load('../models/embedding_model.model')
embedding = EmbeddingRecommender()
embedding.model = embedding_model

# Load hybrid config (weights)
with open('../models/hybrid_config.pkl', 'rb') as f:
    hybrid_data = pickle.load(f)
weights = hybrid_data.get('weights', (0.4, 0.3, 0.3))  # fallback if missing

# Recreate CF recommender object
cf = CFRecommender()
cf.model = cf_model
# You might also need to reload user-item matrix or mappings if your CF uses them:
# cf.user_item_matrix = ... (reload or recompute)
# cf.playlist_map = ... (reload or recompute)

# Recreate hybrid recommender
hybrid = HybridRecommender(cf_model=cf, content_model=content, embedding_model=embedding, weights=weights)
